# **Isolation Forest**

This notebook evaluates Isolation Forest as the unsupervised machine learning model for HDFS anomaly detection.

Unlike Logistic Regression and Random Forest, Isolation Forest does not use class labels during training. Instead, the model is trained only on normal HDFS blocks and learns the characteristics of expected system behavior. Blocks that differ substantially from the learned normal patterns are then identified as anomalous.

Each of the four engineered feature sets is evaluated independently using the same training and testing splits created during model preparation. Model performance is assessed using Accuracy, Precision, Recall, and F1 Score to allow direct comparison with the statistical rule based baseline and supervised learning models.

## Environment Setup

The required libraries are imported, and the project directory structure is initialized. The prepared training and testing datasets generated during model preparation will be loaded from the `data/processed/splits/` directory.

In [1]:
# Import required libraries

from pathlib import Path

import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

In [2]:
# Determine the project root directory

current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

processed_dir = project_root / "data" / "processed"
splits_dir = processed_dir / "splits"

results_dir = project_root / "results"
isolation_forest_results_dir = results_dir / "isolation_forest"

isolation_forest_results_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Splits folder:", splits_dir)
print("Results folder:", isolation_forest_results_dir)

Project root: c:\Users\taman\Documents\hdfs-anomaly-detection-study
Splits folder: c:\Users\taman\Documents\hdfs-anomaly-detection-study\data\processed\splits
Results folder: c:\Users\taman\Documents\hdfs-anomaly-detection-study\results\isolation_forest


## Load Prepared Training and Testing Data

The prepared training and testing datasets are loaded for each engineered feature set. Although Isolation Forest does not use class labels during training, the labels are still loaded so that normal training records can be identified and the final predictions can be evaluated against the ground truth testing labels.

In [3]:
# Load the prepared training and testing datasets for each feature set

isolation_forest_data = {}

for i in range(1, 5):

    feature_name = f"feature_set_{i}"
    feature_dir = splits_dir / feature_name

    isolation_forest_data[feature_name] = {
        "X_train": pd.read_csv(feature_dir / "X_train.csv"),
        "X_test": pd.read_csv(feature_dir / "X_test.csv"),
        "y_train": pd.read_csv(feature_dir / "y_train.csv").squeeze(),
        "y_test": pd.read_csv(feature_dir / "y_test.csv").squeeze(),
    }

In [4]:
# Verify the loaded dataset dimensions

for name, data in isolation_forest_data.items():

    print(f"\n{name}")
    print(f"X_train: {data['X_train'].shape}")
    print(f"X_test : {data['X_test'].shape}")
    print(f"y_train: {data['y_train'].shape}")
    print(f"y_test : {data['y_test'].shape}")


feature_set_1
X_train: (460048, 29)
X_test : (115013, 29)
y_train: (460048,)
y_test : (115013,)

feature_set_2
X_train: (460048, 31)
X_test : (115013, 31)
y_train: (460048,)
y_test : (115013,)

feature_set_3
X_train: (460048, 311)
X_test : (115013, 311)
y_train: (460048,)
y_test : (115013,)

feature_set_4
X_train: (460048, 1082)
X_test : (115013, 1082)
y_train: (460048,)
y_test : (115013,)


## Train Isolation Forest Models

Unlike the supervised learning models, Isolation Forest is trained only on normal HDFS blocks. During training, the model learns the characteristics of expected system behavior without using anomaly labels to guide the learning process.

The anomaly labels are used only to identify and remove anomalous training records before model training. Once the normal training subset has been created, the labels are no longer used until the final evaluation on the testing data.

In [5]:
# Train an Isolation Forest model for each engineered feature set

# Create a dictionary to store the trained Isolation Forest models
isolation_forest_models = {}

# Train one model for each feature set
for name, data in isolation_forest_data.items():

    # Retrieve the training features and labels
    X_train = data["X_train"]
    y_train = data["y_train"]

    # Keep only the normal training records
    X_train_normal = X_train[y_train == 0]

    # Create the Isolation Forest model
    # contamination specifies the expected proportion of anomalies.
    # random_state=42 ensures reproducible results.
    model = IsolationForest(
        contamination="auto",
        random_state=42
    )

    # Train the model using only normal training records
    model.fit(X_train_normal)

    # Store the trained model
    isolation_forest_models[name] = model

In [6]:
# Confirm that all Isolation Forest models were successfully trained

for name in isolation_forest_models:

    print(f"{name}: Model trained successfully")

feature_set_1: Model trained successfully
feature_set_2: Model trained successfully
feature_set_3: Model trained successfully
feature_set_4: Model trained successfully


The first step in training the Isolation Forest model is to create a training dataset containing only normal HDFS blocks. Although the anomaly labels are available, they are used only to identify and remove anomalous training records. By selecting only the records where `y_train == 0`, the model learns the characteristics of normal system behavior without being exposed to anomalous examples during training.

Unlike the supervised learning models, the Isolation Forest algorithm is configured with `contamination="auto"` rather than specifying the expected percentage of anomalies. This allows the model to determine its own decision boundary based solely on the normal training data, preserving the unsupervised nature of the experiment. The anomaly labels are used again only during the final evaluation to compare the model's predictions with the ground-truth testing labels.

## Evaluate Isolation Forest Performance

Each trained Isolation Forest model is evaluated using the corresponding testing dataset.

Isolation Forest returns predictions using `1` for normal observations and `-1` for anomalous observations. These predictions are converted to the label format used throughout this study, where `0` represents normal behavior and `1` represents anomalous behavior.

The converted predictions are then compared with the ground-truth testing labels to calculate Accuracy, Precision, Recall, and F1 Score.

In [8]:
# Evaluate the performance of each Isolation Forest model

# Create a list to store the evaluation results
isolation_forest_results = []

# Evaluate one model for each feature set
for name, model in isolation_forest_models.items():

    # Retrieve the testing features and ground-truth labels
    X_test = isolation_forest_data[name]["X_test"]
    y_test = isolation_forest_data[name]["y_test"]

    # Generate Isolation Forest predictions
    # The model returns 1 for normal records and -1 for anomalies
    raw_predictions = model.predict(X_test)

    # Convert predictions to the project label format
    # Normal: 1 becomes 0
    # Anomaly: -1 becomes 1
    y_pred = pd.Series(raw_predictions, index=X_test.index).map({
        1: 0,
        -1: 1
    })

    # Calculate evaluation metrics
    isolation_forest_results.append({
        "Feature_Set": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
    })

# Convert the results into a DataFrame
isolation_forest_results = pd.DataFrame(isolation_forest_results)

# Display the evaluation results
display(isolation_forest_results)

,Feature_Set,Accuracy,Precision,Recall,F1
0,feature_set_1,0.766644,0.036641,0.275534,0.064680
1,feature_set_2,0.640102,0.057674,0.736045,0.106966
2,feature_set_3,0.871641,0.085606,0.349466,0.137524
3,feature_set_4,0.953849,0.042021,0.026425,0.032446


In [9]:
# Export the Isolation Forest results

isolation_forest_results_path = (
    isolation_forest_results_dir / "isolation_forest_results.csv"
)

isolation_forest_results.to_csv(
    isolation_forest_results_path,
    index=False
)

print("Isolation Forest results saved to:")
print(isolation_forest_results_path)

Isolation Forest results saved to:
c:\Users\taman\Documents\hdfs-anomaly-detection-study\results\isolation_forest\isolation_forest_results.csv


# Isolation Forest Summary

Isolation Forest was successfully trained and evaluated across all four engineered feature sets. Unlike the supervised learning models, Isolation Forest was trained using only normal HDFS blocks, allowing the model to learn expected system behavior without using anomaly labels during training.

Among the four feature sets, **Feature Set 3** produced the highest F1 score (**0.1375**). Although this represents an improvement over the simpler feature sets, Isolation Forest performed substantially worse than both Logistic Regression and Random Forest, as well as the statistical rule based baseline. These results suggest that learning normal behavior alone was insufficient to accurately distinguish anomalous HDFS blocks within this dataset.

Unlike the supervised models, the inclusion of two event sequence features provided a measurable improvement in performance. However, extending the feature representation to include three event sequences resulted in a decrease in performance, indicating that the additional engineered features introduced greater complexity without improving anomaly detection.

Overall, Isolation Forest demonstrated the weakest performance among the evaluated approaches. These results indicate that, for the HDFS dataset, supervised learning methods benefit significantly from labeled anomaly examples and are considerably more effective than an unsupervised approach trained solely on normal system behavior.